# Prediksi Dini Mahasiswa Non-Aktif
**Parsimonious Logistic Regression versus XGBoost untuk Prediksi Dini Mahasiswa Non-Aktif dengan Rolling-Origin Validation**

Notebook pendamping naskah UTS Advanced Machine Learning (Magister Ilmu Komputer, Universitas Nusa Mandiri). Alur mengikuti enam fase CRISP-DM:
Business Understanding, Data Understanding, Data Preparation, Modeling, Evaluation, dan Deployment.

**Tentang data.** Data mentah tidak disertakan karena memuat catatan akademik mahasiswa sebuah perguruan tinggi swasta di Indonesia.
Struktur kolom dijelaskan di `README.md`. Untuk menjalankan notebook, letakkan file `tabel_target1_nonaktif_v2.csv` di folder yang sama atau di `data/`.
Notebook sengaja tidak menampilkan baris data individual; semua keluaran berupa ringkasan, tabel metrik, dan gambar.

**Kaitan dengan naskah**

| Naskah | Bagian notebook |
|---|---|
| Tabel 2 (konfigurasi model) | 3.2, 3.3, 4.1 |
| Tabel 3 (28 fitur) | 3.1 |
| Tabel 4 (metrik top 10%, ROC-AUC, PR-AUC, latensi) | 5.2, 5.14 |
| Tabel 5 (skema validasi) | 5.6 |
| Tabel 6 (odds ratio) | 5.11 |
| Tabel 7 (analisis kesalahan) | 5.12 |
| Gambar 3 (PR-AUC per semester dan per skema validasi) | 5.4, 5.6, 5.6b |
| Gambar 4 (kurva konvergensi XGBoost) | 5.15 |
| Gambar 5 (confusion matrix) | 5.16 |
| Gambar 6 (SHAP summary plot) | 5.16 |
| Gambar 7 (reliability curve) | 5.9 |

Waktu jalan total sekitar 15 menit di Google Colab (CPU). Nilai latensi dapat sedikit berbeda antarmesin.


## 0. Persiapan
Memuat pustaka dan data. Di Google Colab semua pustaka sudah tersedia; di komputer lokal jalankan `pip install -r requirements.txt`.

In [ ]:
import os, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import shap, statsmodels.api as sm
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from xgboost import XGBClassifier
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)
plt.rcParams.update({'font.size': 9})
print('Pustaka siap.')

In [ ]:
NAMA_FILE = 'tabel_target1_nonaktif_v2.csv'
lokasi = [p for p in [NAMA_FILE, os.path.join('data', NAMA_FILE)] if os.path.exists(p)]
if not lokasi:
    try:
        from google.colab import files
        files.upload()
        lokasi = [NAMA_FILE]
    except ImportError:
        raise FileNotFoundError('Data tidak disertakan di repositori. Letakkan ' + NAMA_FILE + ' di folder ini atau di data/.')
df = pd.read_csv(lokasi[0])
df['kode_prodi'] = df['kode_prodi'].astype(str)
print('Data dimuat:', df.shape)

## 1. Business Understanding
Prediksi dibuat sehari sebelum masa KRS semester *t* dibuka. *Risk set* adalah mahasiswa aktif atau MBKM pada semester *t*−1.
Label 1 berarti mahasiswa tersebut non-aktif pada semester *t*. Dosen PA diasumsikan hanya sanggup menghubungi 10% mahasiswa per semester,
sehingga metrik utama adalah PR-AUC dan Recall@10%.

In [ ]:
TARGET = 'y_nonaktif'        # label: non-aktif pada semester t
PERSEN_DAFTAR = 0.10         # kapasitas dosen PA (top 10% per semester)
MULAI_UJI = '2022-2'         # semester uji pertama pada rolling-origin validation
SEED = 42

## 2. Data Understanding

### 2.1 Gambaran umum data

In [ ]:
print('Jumlah baris       :', len(df))
print('Jumlah mahasiswa   :', df.kode_mhs.nunique())
print('Kejadian non-aktif :', int(df[TARGET].sum()), f'({df[TARGET].mean():.2%})')
# Ringkasan kolom (tanpa menampilkan baris data individual)
ringkas = pd.DataFrame({'tipe': df.dtypes.astype(str), 'persen_kosong': (df.isna().mean() * 100).round(1)})
display(ringkas.T)

### 2.2 Jumlah baris dan kejadian per semester
Dipakai pada Bab III bagian C dan Gambar 2.

In [ ]:
tabel_semester = df.groupby('periode').agg(baris=(TARGET, 'size'), kejadian=(TARGET, 'sum'))
tabel_semester['persen'] = (tabel_semester.kejadian / tabel_semester.baris * 100).round(2)
display(tabel_semester)

## 3. Data Preparation

### 3.1 Jenis kolom (Tabel 3)
Kolom identitas dan label dipisahkan dari fitur. Fitur dasar berjumlah 28 (26 angka dan 2 kategori). Fitur riwayat (11) hanya dipakai pada analisis tambahan.

In [ ]:
BUKAN_FITUR = ['kode_mhs', 'periode', TARGET, 'y_nonaktif_smt_depan']
KATEGORI = ['status_l1', 'kode_prodi']
RIWAYAT = [c for c in df.columns if c.endswith('_riwayat')]
ANGKA = [c for c in df.columns if c not in BUKAN_FITUR + KATEGORI + RIWAYAT]
KANDIDAT = ANGKA + KATEGORI

print('Fitur angka    :', len(ANGKA))
print('Fitur kategori :', len(KATEGORI))
print('Fitur dasar    :', len(KANDIDAT))
print('Fitur riwayat  :', len(RIWAYAT))

### 3.2 Kumpulan fitur (Tabel 2)
Tiga kumpulan ditetapkan di awal. Kumpulan keempat dipilih oleh *nested feature selection* di setiap putaran (Bagian 3.4).

In [ ]:
SATU_FITUR = ['sks_lulus_smt_l1']                       # baseline
TIGA_FITUR = ['sks_lulus_smt_l1', 'ips_l1', 'ipk_l1']   # domain knowledge
# KANDIDAT = 28 fitur dasar

### 3.3 Preprocessing
Logistic Regression: *median imputation* + *missing indicator*, standardisasi, *one-hot encoding*. XGBoost membaca nilai *missing* langsung. Semua langkah berada di dalam *pipeline*, sehingga dihitung ulang dari *training set* setiap putaran.

In [ ]:
def model_lr(fitur, C=1.0, penalty='l2'):
    angka = [f for f in fitur if f in ANGKA + RIWAYAT]
    kat = [f for f in fitur if f in KATEGORI]
    bagian = []
    if angka:
        bagian.append(('angka', make_pipeline(SimpleImputer(strategy='median', add_indicator=True),
                                              StandardScaler()), angka))
    if kat:
        bagian.append(('kategori', OneHotEncoder(handle_unknown='ignore'), kat))
    solver = 'liblinear' if penalty == 'l1' else 'lbfgs'
    return make_pipeline(ColumnTransformer(bagian),
                         LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=3000))

def model_xgb(fitur):
    angka = [f for f in fitur if f not in KATEGORI]
    kat = [f for f in fitur if f in KATEGORI]
    return make_pipeline(
        ColumnTransformer([('angka', 'passthrough', angka),
                           ('kategori', OneHotEncoder(handle_unknown='ignore'), kat)]),
        XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, eval_metric='logloss', random_state=SEED, n_jobs=-1))
print('Pipeline siap.')

### 3.4 Nested feature selection
Fitur diurutkan dengan Lasso dan SHAP pada *inner training set*. Metode dan jumlah fitur k dipilih dari PR-AUC tertinggi pada *validation set* (semester latih terakhir). *Test set* tidak pernah dipakai.

In [ ]:
def ke_fitur_asli(nama_kolom):
    bagian, nama = nama_kolom.split('__', 1)
    if bagian == 'kategori':
        return [f for f in KATEGORI if nama.startswith(f + '_')][0]
    return nama.replace('missingindicator_', '')

GRID_C = np.logspace(-4, 1, 40)

def urutan_lasso(latih):
    masuk = {}
    for c in GRID_C:
        m = model_lr(KANDIDAT, C=c, penalty='l1').fit(latih[KANDIDAT], latih[TARGET])
        for n, b in zip(m[0].get_feature_names_out(), m[-1].coef_[0]):
            if b != 0:
                masuk.setdefault(ke_fitur_asli(n), c)
    return sorted(KANDIDAT, key=lambda f: masuk.get(f, np.inf))

def urutan_shap(latih):
    m = model_xgb(KANDIDAT).fit(latih[KANDIDAT], latih[TARGET])
    X = m[0].transform(latih[KANDIDAT])
    X = X.toarray() if hasattr(X, 'toarray') else X
    nilai = np.abs(shap.TreeExplainer(m[-1]).shap_values(X)).mean(0)
    per_fitur = pd.Series(nilai, index=[ke_fitur_asli(n) for n in m[0].get_feature_names_out()]).groupby(level=0).sum()
    return list(per_fitur.reindex(KANDIDAT).fillna(0).sort_values(ascending=False).index)

METODE_URUTAN = {'SHAP': urutan_shap, 'Lasso': urutan_lasso}

def pilih_fitur_bersarang(latih):
    p_val = sorted(latih.periode.unique())[-1]
    dalam, val = latih[latih.periode < p_val], latih[latih.periode == p_val]
    skor = {}
    for metode, urutkan in METODE_URUTAN.items():
        urutan = urutkan(dalam)
        for k in range(1, len(KANDIDAT) + 1):
            f = urutan[:k]
            m = model_lr(f).fit(dalam[f], dalam[TARGET])
            skor[(metode, k)] = average_precision_score(val[TARGET], m.predict_proba(val[f])[:, 1])
    metode, k = max(skor, key=lambda x: (round(skor[x], 6), -x[1]))
    fitur = METODE_URUTAN[metode](latih)[:k]
    info = {'Semester validasi': p_val, 'Kejadian validasi': int(val[TARGET].sum()),
            'Metode': metode, 'k': k, 'PR-AUC validasi': round(skor[(metode, k)], 3), 'Fitur': ', '.join(fitur)}
    return fitur, info
print('Fungsi nested feature selection siap.')

## 4. Modeling

### 4.1 Daftar model
Setiap model adalah pasangan algoritma dan kumpulan fitur.

In [ ]:
def pakai_lr(fitur, C=1.0):
    def jalankan(latih, uji):
        m = model_lr(fitur, C).fit(latih[fitur], latih[TARGET])
        return m.predict_proba(uji[fitur])[:, 1]
    return jalankan

def pakai_xgb(fitur):
    def jalankan(latih, uji):
        m = model_xgb(fitur).fit(latih[fitur], latih[TARGET])
        return m.predict_proba(uji[fitur])[:, 1]
    return jalankan

pilihan_bersarang = []
def pakai_bersarang(latih, uji):
    fitur, info = pilih_fitur_bersarang(latih)
    pilihan_bersarang.append({'Semester uji': uji.periode.iloc[0], **info})
    m = model_lr(fitur).fit(latih[fitur], latih[TARGET])
    return m.predict_proba(uji[fitur])[:, 1]

LR1, LR3, LR28 = 'Logistic Regression 1 fitur', 'Logistic Regression 3 fitur', 'Logistic Regression 28 fitur'
XGB, NFS = 'XGBoost 28 fitur', 'Logistic Regression nested feature selection'
MODEL = {LR1: pakai_lr(SATU_FITUR), LR3: pakai_lr(TIGA_FITUR), LR28: pakai_lr(KANDIDAT, C=0.1),
         XGB: pakai_xgb(KANDIDAT), NFS: pakai_bersarang}
DAFTAR_MODEL = list(MODEL)
DAFTAR_MODEL

### 4.2 Fungsi rolling-origin validation
Untuk setiap semester uji, model dilatih dengan seluruh semester sebelumnya lalu memprediksi semester tersebut.

In [ ]:
def uji_bergulir(data, daftar, tampilkan=True):
    keluar = []
    for p in sorted(data.periode.unique()):
        if p < MULAI_UJI:
            continue
        latih, uji = data[data.periode < p], data[data.periode == p]
        k = uji[['kode_mhs', 'periode', 'kode_prodi', 'angkatan', TARGET]].copy()
        mulai = time.time()
        for nama, jalankan in daftar.items():
            k[nama] = jalankan(latih, uji)
        keluar.append(k)
        if tampilkan:
            print(f'{p}: latih {len(latih):5d} | uji {len(uji):4d} | kejadian {int(uji[TARGET].sum()):2d} | {time.time() - mulai:.0f} detik')
    return pd.concat(keluar, ignore_index=True)

### 4.3 Menjalankan semua model
Bagian ini memerlukan sekitar 10–15 menit karena *nested feature selection*.

In [ ]:
pilihan_bersarang.clear()
hasil = uji_bergulir(df, MODEL)
PERIODE_UJI = sorted(hasil.periode.unique())
print('Total baris uji:', len(hasil), '| kejadian:', int(hasil[TARGET].sum()))

## 5. Evaluation

### 5.1 Fungsi metrik

In [ ]:
def ambil_teratas(g, kolom, persen):
    return g.nlargest(max(1, int(np.ceil(len(g) * persen))), kolom)

def recall_teratas(data, kolom, persen=PERSEN_DAFTAR):
    return np.mean([ambil_teratas(g, kolom, persen)[TARGET].sum() / g[TARGET].sum()
                    for _, g in data.groupby('periode') if g[TARGET].sum() > 0])

def presisi_teratas(data, kolom, persen=PERSEN_DAFTAR):
    return np.mean([ambil_teratas(g, kolom, persen)[TARGET].mean() for _, g in data.groupby('periode')])

def prauc(data, kolom):
    return pd.Series({k: average_precision_score(data[TARGET], data[k]) for k in kolom})

def uji_selisih(data, pasangan, B=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    data = data.reset_index(drop=True)
    mhs = data.kode_mhs.unique(); idx = data.groupby('kode_mhs').indices
    selisih = {f'{a} − {b}': [] for a, b in pasangan}
    for _ in range(B):
        s = data.iloc[np.concatenate([idx[m] for m in rng.choice(mhs, size=len(mhs), replace=True)])]
        for a, b in pasangan:
            selisih[f'{a} − {b}'].append(average_precision_score(s[TARGET], s[a]) - average_precision_score(s[TARGET], s[b]))
    t = pd.DataFrame({k: {'Selisih': np.mean(v), 'Batas bawah 95%': np.percentile(v, 2.5),
                          'Batas atas 95%': np.percentile(v, 97.5)} for k, v in selisih.items()}).T.round(3)
    t['Bermakna?'] = np.where((t['Batas bawah 95%'] > 0) | (t['Batas atas 95%'] < 0), 'ya', 'tidak')
    return t

### 5.2 Kinerja model pada rolling-origin validation (ROC-AUC dan PR-AUC pada Tabel 4)

In [ ]:
y = hasil[TARGET]
tabel_kinerja = pd.DataFrame([{
    'Model': n, 'PR-AUC': average_precision_score(y, hasil[n]), 'ROC-AUC': roc_auc_score(y, hasil[n]),
    'Recall@10%': recall_teratas(hasil, n), 'Precision@10%': presisi_teratas(hasil, n),
    'Brier score': brier_score_loss(y, hasil[n])} for n in DAFTAR_MODEL]).set_index('Model').round(3)
print('PR-AUC random classifier:', round(y.mean(), 3))
display(tabel_kinerja)

### 5.3 Selisih PR-AUC dan 95% confidence interval
Bootstrap per mahasiswa, 1.000 kali. Selisih bermakna bila interval tidak memuat nol. Dipakai pada pembahasan Tabel 4 dan Tabel 5.

In [ ]:
tabel_selisih = uji_selisih(hasil, [(LR3, LR1), (LR28, LR3), (XGB, LR3), (XGB, LR28),
                                    (NFS, LR3), (NFS, LR28), (NFS, XGB)])
display(tabel_selisih)

### 5.4 PR-AUC per semester uji (Gambar 3a)

In [ ]:
per_semester = hasil.groupby('periode').apply(lambda g: prauc(g, DAFTAR_MODEL))
display(per_semester.round(3))

gaya = {LR1: dict(marker='v', color='0.6', ls=':'), LR3: dict(marker='o', color='black', lw=2),
        LR28: dict(marker='s', color='0.45', ls='--'), XGB: dict(marker='^', color='0.2', ls='-.'),
        NFS: dict(marker='D', color='0.55', mfc='white')}
fig, ax = plt.subplots(figsize=(9, 4))
for n in DAFTAR_MODEL:
    ax.plot(per_semester.index, per_semester[n], label=n, ms=5, **gaya[n])
ax.set_xlabel('Semester uji'); ax.set_ylabel('PR-AUC'); ax.set_ylim(0, 0.8)
ax.grid(alpha=.3, color='0.7'); ax.legend(fontsize=8, ncol=2, loc='lower right')
plt.tight_layout(); plt.savefig('gambar3a_per_semester.png', dpi=200); plt.show()

### 5.5 Pilihan nested feature selection per putaran
Metode peringkat dan jumlah fitur k yang terpilih di setiap putaran (dibahas di Bab IV dan Bab V).

In [ ]:
tabel_pilihan = pd.DataFrame(pilihan_bersarang).set_index('Semester uji')
display(tabel_pilihan)
print('Frekuensi metode:', tabel_pilihan.Metode.value_counts().to_dict())

### 5.6 Random cross-validation vs rolling-origin validation (Tabel 5, Gambar 3b)
Model yang sama diuji dengan *stratified* 5-fold per baris dan *group* 5-fold per mahasiswa pada data 2022-2 sampai 2025-2.

In [ ]:
def uji_acak(data, pembagi, kelompok, daftar):
    data = data.reset_index(drop=True)
    k = data[['kode_mhs', 'periode', TARGET]].copy()
    for n in daftar: k[n] = 0.0
    for tr, te in pembagi.split(data, data[TARGET], kelompok):
        for n, jalankan in daftar.items():
            k.loc[te, n] = jalankan(data.iloc[tr], data.iloc[te])
    return k

TIGA_MODEL = {n: MODEL[n] for n in [LR3, LR28, XGB]}
data_uji = df[df.periode >= MULAI_UJI].reset_index(drop=True)
acak_baris = uji_acak(data_uji, StratifiedKFold(5, shuffle=True, random_state=0), None, TIGA_MODEL)
acak_mhs = uji_acak(data_uji, GroupKFold(5), data_uji.kode_mhs, TIGA_MODEL)

cara = {'Random CV per baris': acak_baris, 'Random CV per mahasiswa': acak_mhs, 'Rolling-origin validation': hasil}
tabel_cara = pd.DataFrame({c: prauc(h, list(TIGA_MODEL)) for c, h in cara.items()}).T.round(3)
tabel_cara_ci = pd.concat({c: uji_selisih(h, [(XGB, LR3)]).iloc[0] for c, h in cara.items()}, axis=1).T
display(tabel_cara); display(tabel_cara_ci)

fig, ax = plt.subplots(figsize=(9, 4)); x = np.arange(3); w = 0.26
for i, (n, fc, h) in enumerate(zip(tabel_cara.columns, ['black', '0.6', 'white'], ['', '', '///'])):
    b = ax.bar(x + (i - 1) * w, tabel_cara[n], w, label=n, color=fc, edgecolor='black', hatch=h)
    ax.bar_label(b, fmt='%.3f', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(tabel_cara.index); ax.set_ylabel('PR-AUC'); ax.set_ylim(0, 0.7)
ax.grid(alpha=.3, axis='y', color='0.7'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('gambar3b_cara_uji.png', dpi=200); plt.show()

### 5.6b Gambar 3 gabungan (seperti di naskah)
Panel (a) PR-AUC per semester uji, panel (b) PR-AUC menurut skema validasi.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.2, 2.9), gridspec_kw={'width_ratios': [1.35, 1]})
for n in DAFTAR_MODEL:
    a1.plot(per_semester.index, per_semester[n], ms=4, label=n, **gaya[n])
a1.set_ylabel('PR-AUC'); a1.set_xlabel('Semester uji'); a1.set_ylim(0, 0.8); a1.grid(alpha=.3)
a1.tick_params(axis='x', labelsize=7); a1.set_title('(a) PR-AUC per semester uji', fontsize=8)
a1.legend(fontsize=6, loc='lower right', frameon=False)
label = {'Random CV per baris': 'Random CV\nper baris', 'Random CV per mahasiswa': 'Random CV\nper mahasiswa',
         'Rolling-origin validation': 'Rolling-origin\nvalidation'}
x = np.arange(3); w = 0.26
for i, (n, fc, hh) in enumerate(zip([LR3, LR28, XGB], ['black', '0.6', 'white'], ['', '', '////'])):
    b = a2.bar(x + (i - 1) * w, tabel_cara[n], w, color=fc, edgecolor='black', hatch=hh, lw=0.6, label=n)
    a2.bar_label(b, fmt='%.2f', fontsize=5.5, padding=1)
a2.set_xticks(x); a2.set_xticklabels([label[c] for c in tabel_cara.index], fontsize=6.8)
a2.set_ylim(0, 0.85); a2.set_ylabel('PR-AUC'); a2.grid(alpha=.3, axis='y')
a2.legend(fontsize=6, frameon=False, loc='upper right'); a2.set_title('(b) PR-AUC menurut cara pengujian', fontsize=8)
plt.tight_layout(); plt.savefig('gambar3_prauc.png', dpi=300); plt.show()


### 5.7 Analisis tambahan: penambahan 11 fitur riwayat (tidak ditampilkan di naskah)
Logistic Regression dengan 28 fitur dasar + 11 fitur riwayat dibandingkan dengan Logistic Regression 3 fitur.

In [ ]:
LR39 = 'Logistic Regression 39 fitur'
hasil_riwayat = uji_bergulir(df, {LR3: MODEL[LR3], LR39: pakai_lr(KANDIDAT + RIWAYAT, C=0.1)}, tampilkan=False)
display(prauc(hasil_riwayat, [LR3, LR39]).round(3).rename('PR-AUC').to_frame())
tabel_riwayat = uji_selisih(hasil_riwayat, [(LR39, LR3)])
display(tabel_riwayat)

### 5.8 Global feature importance SHAP
XGBoost dilatih dengan semua semester sebelum semester uji terakhir, lalu dihitung rata-rata |SHAP| pada semester uji terakhir. Gambarnya ada di 5.16.

In [ ]:
p_akhir = PERIODE_UJI[-1]
latih, uji = df[df.periode < p_akhir], df[df.periode == p_akhir]
m = model_xgb(KANDIDAT).fit(latih[KANDIDAT], latih[TARGET])
X_uji = pd.DataFrame(m[0].transform(uji[KANDIDAT]), columns=[n.split('__')[-1] for n in m[0].get_feature_names_out()])
nilai_shap = shap.TreeExplainer(m[-1]).shap_values(X_uji)
tabel_shap = pd.Series(np.abs(nilai_shap).mean(0), index=X_uji.columns).sort_values(ascending=False).head(12).round(4)
display(tabel_shap.rename('rata-rata |SHAP|').to_frame())

### 5.9 Reliability curve (Gambar 7)

In [ ]:
BATAS = [0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50, 1.00]
TIGA = [LR3, LR28, XGB]
gaya_k = {LR3: dict(marker='o', color='black'), LR28: dict(marker='s', color='0.5', ls='--'),
          XGB: dict(marker='^', color='0.25', ls='-.')}
tabel_kalibrasi = {}
fig, ax = plt.subplots(figsize=(3.6, 3.0))
g = np.linspace(0, 1, 400); ax.plot(g, g, 'k:', lw=0.9, label='Perfect calibration')
for n in TIGA:
    kel = pd.cut(hasil[n], BATAS, include_lowest=True)
    t = pd.DataFrame({'p': hasil[n], 'y': y}).groupby(kel, observed=True).agg(
        jumlah=('y', 'size'), peluang_rata2=('p', 'mean'), kejadian_nyata=('y', 'mean'))
    tabel_kalibrasi[n] = t
    ax.plot(t.peluang_rata2, t.kejadian_nyata, label=n, ms=4, lw=1, **gaya_k[n])
ax.set_xscale('symlog', linthresh=0.01); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Rata-rata peluang prediksi'); ax.set_ylabel('Proporsi non-aktif sebenarnya')
ax.grid(alpha=.3); ax.legend(fontsize=6, frameon=False, loc='upper left')
plt.tight_layout(); plt.savefig('gambar7_kalibrasi.png', dpi=300); plt.show()
for n in TIGA:
    print(n); display(tabel_kalibrasi[n].round(3))


### 5.10 Simulasi threshold intervensi
Dibahas pada pembahasan Gambar 7 (ambang 0,30).

In [ ]:
baris = []
for n in TIGA:
    for a in [0.30, 0.50]:
        tanda = hasil[n] >= a; tp = int((tanda & (y == 1)).sum())
        baris.append({'Model': n, 'Aturan': f'threshold {a:.2f}', 'Ditandai': int(tanda.sum()),
                      'Per semester': round(tanda.groupby(hasil.periode).sum().mean(), 1), 'True positive': tp,
                      'Recall': tp / y.sum(), 'Precision': tp / tanda.sum()})
    atas = pd.concat([ambil_teratas(g, n, PERSEN_DAFTAR) for _, g in hasil.groupby('periode')])
    tp = int(atas[TARGET].sum())
    baris.append({'Model': n, 'Aturan': 'top 10%', 'Ditandai': len(atas), 'Per semester': round(len(atas) / len(PERIODE_UJI), 1),
                  'True positive': tp, 'Recall': tp / y.sum(), 'Precision': tp / len(atas)})
tabel_threshold = pd.DataFrame(baris).set_index(['Model', 'Aturan']).round(3)
display(tabel_threshold)

grid = np.round(np.arange(0.05, 0.81, 0.05), 2)
rec = [((hasil[LR3] >= a) & (y == 1)).sum() / y.sum() for a in grid]
pre = [hasil.loc[hasil[LR3] >= a, TARGET].mean() for a in grid]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(grid, rec, 'o-', color='black', label='Recall'); ax.plot(grid, pre, 's--', color='0.5', label='Precision')
for a in [0.30, 0.50]: ax.axvline(a, color='black', ls=':', lw=1)
ax.set_xlabel('Threshold'); ax.set_ylabel('Nilai'); ax.set_ylim(0, 1); ax.grid(alpha=.3, color='0.7'); ax.legend()
plt.tight_layout(); plt.savefig('tambahan_threshold.png', dpi=200); plt.show()

### 5.11 Odds ratio Logistic Regression 3 fitur (Tabel 6)
Dilatih ulang dengan seluruh data tanpa standardisasi agar odds ratio terbaca per satuan aslinya.

In [ ]:
X = sm.add_constant(df[TIGA_FITUR].fillna(df[TIGA_FITUR].median()))
logit = sm.Logit(df[TARGET], X).fit(disp=0)
ci = logit.conf_int()
tabel_or = pd.DataFrame({'Koefisien': logit.params, 'Odds ratio': np.exp(logit.params),
                         'Batas bawah 95%': np.exp(ci[0]), 'Batas atas 95%': np.exp(ci[1]),
                         'p-value': logit.pvalues}).drop('const').round(4)
display(tabel_or)

profil = pd.DataFrame([[20, 3.5, 3.4], [12, 2.5, 2.9], [6, 1.5, 2.4], [0, 0.0, 2.0]], columns=TIGA_FITUR)
profil['Peluang non-aktif'] = logit.predict(sm.add_constant(profil[TIGA_FITUR], has_constant='add')).round(3)
display(profil)

### 5.12 Analisis kesalahan (Tabel 7)
Median karakteristik *true positive*, *false negative*, dan *false positive* pada top 10% Logistic Regression 3 fitur.

In [ ]:
h = hasil.merge(df[['kode_mhs', 'periode', 'ips_l1', 'sks_lulus_smt_l1', 'ipk_l1']], on=['kode_mhs', 'periode'])
h['masuk_daftar'] = 0
for _, g in h.groupby('periode'):
    h.loc[ambil_teratas(g, LR3, PERSEN_DAFTAR).index, 'masuk_daftar'] = 1
h['kelompok'] = np.select([(h[TARGET] == 1) & (h.masuk_daftar == 1), (h[TARGET] == 1) & (h.masuk_daftar == 0),
                           (h[TARGET] == 0) & (h.masuk_daftar == 1)],
                          ['True positive', 'False negative', 'False positive'], 'Lainnya')
tabel_kesalahan = (h[h.kelompok != 'Lainnya'].groupby('kelompok')
                   .agg(jumlah=('kode_mhs', 'size'), ips_l1=('ips_l1', 'median'),
                        sks_lulus_smt_l1=('sks_lulus_smt_l1', 'median'), ipk_l1=('ipk_l1', 'median')).round(2))
display(tabel_kesalahan)

### 5.13 Recall@10% per program studi dan angkatan
Dibahas pada bagian analisis kesalahan dan ancaman validitas.

In [ ]:
def recall_kelompok(data, kolom, kelompok):
    data = data.copy(); data['masuk_daftar'] = 0
    for _, g in data.groupby('periode'):
        data.loc[ambil_teratas(g, kolom, PERSEN_DAFTAR).index, 'masuk_daftar'] = 1
    return (data[data[TARGET] == 1].groupby(kelompok)
            .agg(jumlah_nonaktif=('masuk_daftar', 'size'), tertangkap=('masuk_daftar', 'sum'),
                 recall=('masuk_daftar', 'mean')).round(3))

tabel_kelompok = {kel: recall_kelompok(hasil, LR3, kel) for kel in ['kode_prodi', 'angkatan']}
for kel, t in tabel_kelompok.items():
    print(f'Recall@10% {LR3} per {kel}'); display(t)

### 5.14 Metrik klasifikasi pada top 10%, waktu latih, dan latensi (Tabel 4)
Accuracy, precision, recall, dan macro F1 dihitung dari aturan daftar 10% teratas per semester. Latensi diukur sebagai median 30 kali prediksi pada semester uji terakhir, dinyatakan dalam milidetik per 1.000 baris.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
for n in DAFTAR_MODEL:
    flag = pd.Series(0, index=hasil.index)
    for _, g in hasil.groupby('periode'):
        flag[ambil_teratas(g, n, PERSEN_DAFTAR).index] = 1
    hasil['flag_' + n] = flag

latih, uji = df[df.periode < p_akhir], df[df.periode == p_akhir]
fitur_nfs = tabel_pilihan.loc[p_akhir, 'Fitur'].split(', ')
spesifikasi = {LR1: (SATU_FITUR, 1.0), LR3: (TIGA_FITUR, 1.0), LR28: (KANDIDAT, 0.1), XGB: (KANDIDAT, None), NFS: (fitur_nfs, 1.0)}
baris = []
for n, (f, C) in spesifikasi.items():
    mulai = time.perf_counter()
    if n == NFS:
        pilih_fitur_bersarang(latih)
    m = (model_xgb(f) if n == XGB else model_lr(f, C)).fit(latih[f], latih[TARGET])
    waktu_latih = time.perf_counter() - mulai
    ulang = []
    for _ in range(30):
        t0 = time.perf_counter(); m.predict_proba(uji[f]); ulang.append(time.perf_counter() - t0)
    flag = hasil['flag_' + n]
    baris.append({'Model': n, 'Accuracy': accuracy_score(y, flag), 'Precision': precision_score(y, flag),
                  'Recall': recall_score(y, flag), 'Macro F1': f1_score(y, flag, average='macro'),
                  'ROC-AUC': roc_auc_score(y, hasil[n]), 'PR-AUC': average_precision_score(y, hasil[n]),
                  'Waktu latih (detik)': waktu_latih,
                  'Latensi (ms/1.000 baris)': np.median(ulang) / len(uji) * 1e6})
tabel_komparasi = pd.DataFrame(baris).set_index('Model').round(3)
display(tabel_komparasi)

### 5.15 Kurva konvergensi XGBoost (Gambar 4)
Dibuat pada pembagian dalam putaran terakhir: latih sebelum semester validasi, validasi pada semester latih terakhir.

In [ ]:
p_val = sorted(df[df.periode < p_akhir].periode.unique())[-1]
dalam, val = df[df.periode < p_val], df[df.periode == p_val]
m = model_xgb(KANDIDAT)
pra = m[0].fit(dalam[KANDIDAT])
Xd, Xv = pra.transform(dalam[KANDIDAT]), pra.transform(val[KANDIDAT])
xgbm = m[-1]; xgbm.set_params(eval_metric=['logloss', 'aucpr'])
xgbm.fit(Xd, dalam[TARGET], eval_set=[(Xd, dalam[TARGET]), (Xv, val[TARGET])], verbose=False)
ev = xgbm.evals_result(); it = np.arange(1, len(ev['validation_0']['logloss']) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.0, 2.6))
for ax, met, judul in [(a1, 'logloss', '(a) Log-loss'), (a2, 'aucpr', '(b) PR-AUC')]:
    ax.plot(it, ev['validation_0'][met], color='black', lw=1.2, label='Training')
    ax.plot(it, ev['validation_1'][met], color='0.5', lw=1.2, ls='--', label='Validation')
    ax.set_xlabel('Jumlah pohon (iterasi boosting)'); ax.set_title(judul, fontsize=8); ax.grid(alpha=.3); ax.legend(frameon=False)
a1.set_ylabel('Log-loss'); a2.set_ylabel('PR-AUC')
plt.tight_layout(); plt.savefig('gambar4_konvergensi.png', dpi=300); plt.show()
print('Log-loss validasi terendah:', round(min(ev['validation_1']['logloss']), 3), 'pada pohon ke-', int(np.argmin(ev['validation_1']['logloss'])) + 1)

### 5.16 Confusion matrix top 10% (Gambar 5) dan SHAP summary plot (Gambar 6)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(5.6, 2.5))
for ax, n, judul in zip(axs, [LR3, XGB], ['(a) ' + LR3, '(b) ' + XGB]):
    cm = confusion_matrix(y, hasil['flag_' + n])
    ax.imshow(cm, cmap='Greys', vmin=0, vmax=cm.max() * 1.6)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i, j]:,}'.replace(',', '.'), ha='center', va='center', fontsize=9,
                    color='white' if cm[i, j] > cm.max() * 0.9 else 'black')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Tidak ditandai', 'Ditandai'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['Aktif', 'Non-aktif'])
    ax.set_xlabel('Prediksi (top 10%)'); ax.set_title(judul, fontsize=8)
axs[0].set_ylabel('Kenyataan')
plt.tight_layout(); plt.savefig('gambar5_confusion.png', dpi=300); plt.show()

from matplotlib.colors import LinearSegmentedColormap
abu = LinearSegmentedColormap.from_list('abu', ['#d9d9d9', '#000000'])
shap.summary_plot(nilai_shap, X_uji, max_display=10, cmap=abu, show=False, plot_size=(5.6, 3.4))
plt.tight_layout(); plt.savefig('gambar6_shap.png', dpi=300, bbox_inches='tight'); plt.show()

## 6. Deployment
Rancangan pemakaian: daftar top 10% mahasiswa berisiko per semester untuk dosen PA, memakai Logistic Regression 3 fitur yang dilatih ulang dengan semua data terbaru.

### 6.1 Ringkasan daftar mahasiswa berisiko (semester uji terakhir)
Daftar per mahasiswa tidak ditampilkan demi privasi; yang ditampilkan hanya jumlah per program studi dan angkatan.

In [ ]:
daftar_pa = ambil_teratas(hasil[hasil.periode == p_akhir], LR3, PERSEN_DAFTAR)
print(f'{len(daftar_pa)} mahasiswa masuk daftar semester {p_akhir}')
print('Rentang peluang dalam daftar:', round(daftar_pa[LR3].min(), 3), 'sampai', round(daftar_pa[LR3].max(), 3))
display(daftar_pa.groupby('kode_prodi').size().rename('jumlah').to_frame().T)
display(daftar_pa.groupby('angkatan').size().rename('jumlah').to_frame().T)


### 6.2 Simpan semua hasil

In [ ]:
NAMA_HASIL = 'hasil_penelitian_crispdm.xlsx'
with pd.ExcelWriter(NAMA_HASIL) as w:
    tabel_semester.to_excel(w, sheet_name='data_per_semester')
    tabel_komparasi.to_excel(w, sheet_name='Tabel4_komparasi')
    tabel_kinerja.to_excel(w, sheet_name='Tabel4_kinerja_tambahan')
    tabel_selisih.to_excel(w, sheet_name='selisih_CI95')
    tabel_cara.to_excel(w, sheet_name='Tabel5_skema_validasi')
    tabel_cara_ci.to_excel(w, sheet_name='Tabel5_selisih_CI95')
    tabel_or.to_excel(w, sheet_name='Tabel6_odds_ratio')
    tabel_kesalahan.to_excel(w, sheet_name='Tabel7_analisis_kesalahan')
    tabel_pilihan.to_excel(w, sheet_name='nested_FS_per_putaran')
    tabel_threshold.to_excel(w, sheet_name='threshold')
    per_semester.round(4).to_excel(w, sheet_name='per_semester')
    tabel_riwayat.to_excel(w, sheet_name='fitur_riwayat')
    tabel_shap.to_frame().to_excel(w, sheet_name='shap_global')
    for i, n in enumerate(TIGA):
        tabel_kalibrasi[n].round(4).to_excel(w, sheet_name=f'kalibrasi_{i + 1}')
    for kel, t in tabel_kelompok.items():
        t.to_excel(w, sheet_name=f'recall_{kel}')
print('Tersimpan:', NAMA_HASIL)

try:
    from google.colab import files
    for f in [NAMA_HASIL, 'gambar3_prauc.png', 'gambar3a_per_semester.png', 'gambar3b_cara_uji.png',
              'gambar7_kalibrasi.png', 'tambahan_threshold.png',
              'gambar4_konvergensi.png', 'gambar5_confusion.png', 'gambar6_shap.png']:
        files.download(f)
except ImportError:
    pass